# Magnetic Resonance Imaging II
 
## Exercise 3
 
---
 
**Prof. Dr. Markus Zimmermann**
 
**Mohamed Wassim Bensalem**
 
---

In Exercises 1 and 2 you built a Gradient Echo (GRE) sequence, where each TR acquires a single phase-encoding line. The total scan time is therefore proportional to the number of lines $n_y$. **Echo Planar Imaging (EPI)** takes a fundamentally different approach: after a single RF excitation, all $n_y / R$ phase-encoding lines are acquired in rapid succession by alternating the readout gradient direction and stepping through k-space with small phase-encoding blips. This makes EPI one of the fastest MRI sequences available, and it is the backbone of functional MRI (fMRI) and diffusion-weighted imaging (DWI).
 
However, EPI's speed comes with a cost. Because all lines are acquired after a single excitation, the readout train is long and the signal accumulates phase from off-resonance effects — local variations in the static magnetic field $B_0$ caused by susceptibility differences at tissue interfaces. This leads to geometric distortions and signal loss that worsen with increasing readout duration. Parallel imaging acceleration with factor $R$ shortens the readout train by a factor of $R$, directly reducing off-resonance artifacts. In this exercise you will see this tradeoff directly: **R=1 produces visible distortions, while R=2 yields a clean image.**
 
The reconstruction pipeline is identical to Exercise 2 — the same operators $\mathbf{C}$, $\mathbf{F}$, $\mathbf{M}$ and the same CG solver — so those are provided here without tasks. Your work is in building the EPI sequence and correctly handling the k-space trajectory it produces.

## Software Imports
 
All required libraries and modules are imported here.

In [ ]:
%matplotlib widget

import os
import sys
from typing import Optional, Tuple

import matplotlib.pyplot as plt
import MRzeroCore as mr0
import numpy as np
import pypulseq as pp
import torch
from tqdm import tqdm

sys.path.insert(0, "/home/jovyan/mri/src")
sys.path.insert(0, "/home/jovyan/shared-folder/Magnetresonanztomographie-2/mri/src")

from mri.conopt.linops import LinearOperator
from mri.parim.analytic import cyclic_head_coil

## System and Sequence Parameters
 
Set the acceleration factor `R` here. All downstream cells — the sequence, the k-space regridding, the mask, and the reconstruction — respond to this single variable. Start with `R = 1` to observe the off-resonance artifacts, then switch to `R = 2` to see how acceleration suppresses them.
 
The echo time `te` is defined as the time from the RF pulse center to the center of k-space, i.e. the moment when the phase-encoding gradient passes through $k_y = 0$. For EPI this is determined by the length of the readout train and cannot always be met exactly — if `te` is shorter than the minimum achievable echo time, the sequence will warn you and use `te_min` instead.

In [ ]:
phantom_path_local = "/home/jovyan/mri-data/mri-datasets/brainweb"
phantom_path_shared = "/home/jovyan/shared-folder/Magnetresonanztomographie-2/mri-data/mri-datasets/brainweb"
seq_filename = "epi.seq"

fov_x, fov_y = 256e-3, 256e-3
n_x, n_y = 128, 128
n_channels = 8
R = 1

flip_angle_deg = 90
te = 25e-3

torch.set_default_device("cpu")

## System and Sequence Object

In [ ]:
system = pp.Opts(
    max_grad=28,
    grad_unit="mT/m",
    max_slew=150,
    slew_unit="T/m/s",
)

seq = pp.Sequence(system)

## Linear Operators

The linear operators are identical to Exercise 2. They are provided here so you can focus on the new sequence content.

In [ ]:
class FourierTransformOperator(LinearOperator):

    def __init__(
        self,
        data_shape: Tuple[int, ...],
        axes: Tuple[int, ...],
        device: Optional[torch.device] = None,
    ):

        self.forward_shape = data_shape
        self.adjoint_shape = data_shape
        self.shape = (
            int(2 * torch.prod(torch.tensor(self.adjoint_shape))),
            int(2 * torch.prod(torch.tensor(self.forward_shape))),
        )
        self.internal_dtype = torch.complex64
        self.dtype = torch.float32
        self.device = device
        self.axes = axes

    def _matvec(self, x: torch.Tensor) -> torch.Tensor:
        x = x.view(self.internal_dtype).reshape(self.forward_shape)
        x = torch.fft.ifftshift(x, dim=self.axes)
        x = torch.fft.fftn(x, dim=self.axes, norm="ortho")
        x = torch.fft.fftshift(x, dim=self.axes)
        return x.view(self.dtype).ravel()

    def _rmatvec(self, x: torch.Tensor) -> torch.Tensor:
        x = x.view(self.internal_dtype).reshape(self.adjoint_shape)
        x = torch.fft.ifftshift(x, dim=self.axes)
        x = torch.fft.ifftn(x, dim=self.axes, norm="ortho")
        x = torch.fft.fftshift(x, dim=self.axes)
        return x.view(self.dtype).ravel()


class ChannelOperator(LinearOperator):

    def __init__(
        self,
        coil_sensitivities: torch.Tensor,
        data_shape: Tuple[int, ...],
        device: Optional[torch.device] = None,
    ):
        nX, nY = data_shape
        nC = coil_sensitivities.shape[0]

        self.coil_sensitivities = coil_sensitivities
        self.device = device
        self.dtype = torch.float32
        self.internal_dtype = torch.complex64
        self.forward_shape = (1, nX, nY)
        self.adjoint_shape = (nC, nX, nY)
        self.shape = (
            int(2 * torch.prod(torch.tensor(self.adjoint_shape))),
            int(2 * torch.prod(torch.tensor(self.forward_shape))),
        )

    def _matvec(self, x: torch.Tensor) -> torch.Tensor:
        x = x.view(self.internal_dtype).reshape(self.forward_shape)
        y = self.coil_sensitivities * x
        return y.view(torch.float32).ravel()

    def _rmatvec(self, x: torch.Tensor) -> torch.Tensor:
        x = x.view(self.internal_dtype).reshape(self.adjoint_shape)
        y = torch.sum(torch.conj(self.coil_sensitivities) * x, dim=0, keepdim=True)
        return y.view(torch.float32).ravel()


class DiagonalOperator(LinearOperator):

    def __init__(
        self,
        diagonal_tensor: torch.Tensor,
        data_shape: Tuple[int, ...],
        device: Optional[torch.device] = None,
    ):
        self.diagonal_tensor = diagonal_tensor
        self.device = device
        self.internal_dtype = torch.complex64
        self.dtype = torch.float32
        self.forward_shape = data_shape
        self.adjoint_shape = data_shape
        self.shape = (
            int(2 * torch.prod(torch.tensor(data_shape))),
            int(2 * torch.prod(torch.tensor(data_shape))),
        )

    def _matvec(self, x: torch.Tensor) -> torch.Tensor:
        x = x.view(self.internal_dtype).reshape(self.forward_shape)
        y = self.diagonal_tensor * x
        return y.view(torch.float32).ravel()

    def _rmatvec(self, x: torch.Tensor) -> torch.Tensor:
        x = x.view(self.internal_dtype).reshape(self.adjoint_shape)
        y = torch.conj(self.diagonal_tensor) * x
        return y.view(torch.float32).ravel()

## Task 1: Sequence Components

In GRE, the readout gradient `gx` always traverses k-space in the same direction. EPI alternates: odd lines go left-to-right (`gx`), even lines go right-to-left (`gx_rev`). A small phase-encoding **blip** `gy_blip` steps k-space upward by $R \cdot \Delta k_y$ between each readout line. The pre-phaser `gx_pre` positions the readout at $-k_{x,\text{max}}$ before the first line, and `gy_pre` positions the phase-encoding at $-k_{y,\text{max}}$ before the first blip.
 
The echo time for EPI is the time from the RF center to the center of the readout train (the line passing through $k_y = 0$). Because the entire readout train follows a single excitation, `te_min` depends on the duration of all readout lines and blips:
 
$$t_{e,\text{min}} = t_\text{RF} + t_\text{pre} + \frac{n_y}{2R} \cdot \left( t_\text{readout} + t_\text{blip} \right)$$
 
where $t_\text{RF}$ is the time from the RF center to the end of the RF pulse, $t_\text{pre}$ is the duration of the pre-phaser block, and the factor $n_y / (2R)$ counts the number of readout lines before the k-space center.
 
**Your tasks:**
 
1. Define `gx` and `gx_rev` — forward and reversed readout gradients covering $n_x \cdot \Delta k_x$ in flat area, with `flat_time = 1.6e-3`.
2. Define `gx_pre` — a pre-phaser that positions the readout at $-k_{x,\text{max}}$, i.e. area $= -\text{gx.area} / 2$.
3. Define `gy_pre` — a pre-phaser that positions the phase-encoding at $-k_{y,\text{max}}$, i.e. area $= -n_y \cdot \Delta k_y / 2$.
4. Define `gy_blip` — a phase-encoding blip with area $= R \cdot \Delta k_y$.
5. Define `adc` to acquire `n_x` samples over `flat_time`, with `delay = gx.rise_time`.
6. Compute `te_min` and `te_delay` as described above. Print a warning if `te < te_min`.

In [ ]:
# Your solution here


delta_kx = 1 / fov_x
delta_ky = 1 / fov_y
raster_eps = 1e-12
flat_time = 1.6e-3

rf = pp.make_sinc_pulse(
    flip_angle=np.deg2rad(flip_angle_deg),
    system=system,
    use="excitation",
)

gx = pp.make_trapezoid(
    channel="x",
    flat_area=n_x * delta_kx,
    flat_time=flat_time,
    system=system,
)

gx_rev = pp.make_trapezoid(
    channel="x",
    flat_area=-n_x * delta_kx,
    flat_time=flat_time,
    system=system,
)

gx_pre = pp.make_trapezoid(
    channel="x",
    area=-gx.area / 2,
    system=system,
)

gy_pre = pp.make_trapezoid(
    channel="y",
    area=-n_y * delta_ky / 2,
    system=system,
)

gy_blip = pp.make_trapezoid(
    channel="y",
    area=R * delta_ky,
    system=system,
)

adc = pp.make_adc(
    num_samples=n_x,
    duration=flat_time,
    delay=gx.rise_time,
    system=system,
)

gx_spoiler = pp.make_trapezoid(
    channel="x",
    area=4 * gx.area,
    system=system,
)

te_min = (
    pp.calc_duration(rf)
    - pp.calc_rf_center(rf)[0]
    + pp.calc_duration(gx_pre)
    + n_y / (2 * R) * (pp.calc_duration(gx) + pp.calc_duration(gy_blip))
)

te_delay = (
    np.ceil((te - te_min - raster_eps) / seq.grad_raster_time) * seq.grad_raster_time
)

if te_delay <= 0:
    print(
        f"Echo time {te} shorter than minimal echo time {te_min:.4f} s "
        f"-> Setting echo time to {te_min:.4f} s"
    )
    te_delay = 0
    te = te_min

## Task 2: EPI Sequence Assembly

In EPI, the entire k-space is acquired after a single RF excitation. The sequence plays:
 
1. RF excitation
2. Pre-phasers `gx_pre` and `gy_pre` — position at $(-k_{x,\text{max}},\ -k_{y,\text{max}})$
3. Optional delay to reach the target echo time
4. The readout loop — alternating `gx` / `gx_rev` with a `gy_blip` after each line
5. A spoiler gradient
The loop runs `n_y // (2 * R)` iterations, each acquiring two lines (one forward, one reversed). With acceleration factor `R`, the blip area is $R \cdot \Delta k_y$, so every other acquired line corresponds to one in the full k-space grid.
 
**Your task:** Assemble the EPI sequence. In each loop iteration, play `gx` with `adc`, then `gy_blip`, then `gx_rev` with `adc`, then `gy_blip` again.

In [ ]:
# Your solution here


seq.add_block(rf)

seq.add_block(gx_pre, gy_pre)

seq.add_block(pp.make_delay(te_delay))

for index in range(n_y // (2 * R)):

    seq.add_block(gx, adc)

    seq.add_block(gy_blip)

    seq.add_block(gx_rev, adc)

    seq.add_block(gy_blip)

seq.add_block(gx_spoiler)

## Verification and Export

In [ ]:
ok, error_report = seq.check_timing()

if ok:
    print("Timing check passed successfully")
else:
    print("Timing check failed. Error listing follows:")
    [print(e) for e in error_report]

plt.figure()
seq.plot()

seq.write(seq_filename, v141_compat=True)

## Phantom and Multi-Channel Simulation

This exercise uses a realistic Brainweb digital brain phantom instead of the simple analytical phantom from Exercise 2. The phantom is loaded, interpolated to the target matrix size, and a single axial slice is selected. Run this cell as-is.

In [ ]:
if os.path.isdir(phantom_path_local):
    phantom_path = phantom_path_local
if os.path.isdir(phantom_path_shared):
    phantom_path = phantom_path_shared

phantom = mr0.VoxelGridPhantom.brainweb(os.path.join(phantom_path, "subject05.npz"))

phantom = phantom.interpolate(n_x, n_y, 32).slices([16])
phantom.size[2] = 0.08

coil_sensitivities_recon = cyclic_head_coil(
    (n_channels, n_x, n_y, 1),
).to(torch.complex64)
coil_sensitivities_recon = coil_sensitivities_recon[:, :, :, 0].flip((1, 2))

coil_sensitivities_sim = cyclic_head_coil(
    (n_channels, n_x, n_y, 1),
    fov_scaling=(phantom.size[0] / fov_x, phantom.size[1] / fov_y, 1),
).to(torch.complex64)
phantom.coil_sens = coil_sensitivities_sim

plt.figure()
phantom.plot()

mr0_seq = mr0.Sequence.import_file(seq_filename)

data = phantom.build()
graph = mr0.compute_graph(mr0_seq, data)
signal = mr0.execute_graph(graph, mr0_seq, data, print_progress=True)

## Task 3: Coil Images

The raw signal has shape $(n_y / R \cdot n_x,\ n_\text{channels})$. As in Exercise 2, reshape it to $(n_\text{channels},\ n_x,\ n_y / R)$, zero-fill to $(n_\text{channels},\ n_x,\ n_y)$ by placing the acquired lines at positions `::R`, and apply the inverse Fourier transform to obtain coil images.
 
However, EPI has a critical difference from GRE: **the readout direction alternates every line**. Even-indexed lines (0, 2, 4, ...) are acquired left-to-right (+$k_x$ direction), while odd-indexed lines (1, 3, 5, ...) are acquired right-to-left (−$k_x$ direction). Think carefully about what this means for how the data should be stored in `data_full` before the Fourier transform is applied.
 
**Your tasks:**
 
1. Reshape and permute `signal` to obtain `data` of shape $(n_\text{channels},\ n_x,\ n_y / R)$.
2. Create `data_full` of shape $(n_\text{channels},\ n_x,\ n_y)$ and place the acquired data into `data_full[:, :, ::R]`.
3. Before applying the Fourier transform, think carefully about the EPI k-space trajectory. Not all lines are acquired in the same direction — how should this affect how you store them in `data_full`?
4. Reconstruct `coil_images` by applying the inverse Fourier transform along axes `(1, 2)`.
5. Display the coil images and coil sensitivity maps in 2×4 grids.

What artifacts do you observe in the coil images? Can you explain where they come from?

In [ ]:
# Your solution here


data = signal.clone()
data = data.reshape((n_y // R, n_x, n_channels)).permute((2, 1, 0))

data_full = torch.zeros((n_channels, n_x, n_y), dtype=data.dtype)
data_full[:, :, ::R] = data

data_full[:, :, R :: 2 * R] = torch.flip(data_full[:, :, R :: 2 * R], (1,))

coil_images = torch.fft.fftshift(
    torch.fft.ifftn(
        torch.fft.ifftshift(data_full, dim=(1, 2)),
        dim=(1, 2),
    ),
    dim=(1, 2),
)

plt.figure()
for i in range(n_channels):
    plt.subplot(2, 4, i + 1)
    plt.imshow(torch.abs(coil_images[i]).numpy().T, cmap="gray")
    plt.title(f"Channel {i + 1}")
plt.suptitle("Coil images")
plt.tight_layout()
plt.show()

plt.figure()
for i in range(n_channels):
    plt.subplot(2, 4, i + 1)
    plt.imshow(torch.abs(coil_sensitivities_recon[i]).numpy().T, cmap="viridis")
    plt.title(f"Channel {i + 1}")
plt.suptitle("Coil sensitivity maps")
plt.tight_layout()
plt.show()

## Task 4: Compose the Forward Operator and Compute Regridded Image

Compose the forward operator $\mathbf{A} = \mathbf{M} \mathbf{F} \mathbf{C}$ and compute the matched filter output $\mathbf{A}^\mathrm{H} \mathbf{d}$, exactly as in Exercise 2. The sampling mask now has ones at positions `::R` along the $k_y$ axis.
 
**Your tasks:**
 
1. Construct the sampling mask `mask` of shape $(n_\text{channels},\ n_x,\ n_y)$ with ones at `::R` along the last axis.
2. Instantiate $\mathbf{F}$, $\mathbf{C}$, and $\mathbf{M}$ and compose `A = M @ F @ C`.
3. Compute `AHd = A.H @ data_full.view(torch.float32).ravel()` and display the magnitude image.

In [ ]:
# Your solution here


mask = torch.zeros((n_channels, n_x, n_y), dtype=torch.complex64)
mask[:, :, ::R] = 1

F = FourierTransformOperator((n_channels, n_x, n_y), (1, 2))
C = ChannelOperator(coil_sensitivities_recon, (n_x, n_y))
M = DiagonalOperator(mask, (n_channels, n_x, n_y))

A = M @ F @ C

AHd = A.H @ data_full.view(torch.float32).ravel()

AHd_image = AHd.view(torch.complex64).reshape((n_x, n_y))

plt.figure()
plt.imshow(torch.abs(AHd_image).numpy().T, cmap="gray")
plt.title(
    f"$\\mathbf{{A}}^\\mathrm{{H}} \\mathbf{{d}}$ — Zero-filled Reconstruction (R={R})"
)
plt.show()

## Task 5: Iterative Reconstruction with Conjugate Gradient

The CG solver is provided below. Run it to recover the image from the undersampled, noisy k-space data.

In [ ]:
def conjugate_gradient(
    Q: LinearOperator,
    b: torch.Tensor,
    x: torch.Tensor,
    maxiter: int = 10,
) -> torch.Tensor:

    g_prev = torch.ones(x.shape, dtype=x.dtype)
    p_prev = torch.zeros(x.shape, dtype=x.dtype)

    for iteration in tqdm(range(maxiter)):

        g = Q @ x - b

        conjugacy_factor = g @ g / (g_prev @ g_prev)
        p = g + conjugacy_factor * p_prev
        stepsize = g @ g / (p @ (Q @ p))
        x = x - stepsize * p

        g_prev = g.clone()
        p_prev = p.clone()

    return x

**Your tasks:**
 
1. Construct $\mathbf{Q} = \mathbf{A}^\mathrm{H} \mathbf{A}$ and set $\mathbf{b} = \mathbf{A}^\mathrm{H} \mathbf{d}$.
2. Initialize $\mathbf{x}$ as a zero tensor of shape $(2 \cdot n_x \cdot n_y,)$ with `dtype=torch.float32`.
3. Run `conjugate_gradient(Q, b, x, maxiter=30)` and display the result alongside the zero-filled image.

In [ ]:
# Your solution here


Q = A.H @ A

b = AHd
x = torch.zeros(n_x * n_y, dtype=torch.complex64).view(torch.float32)

x_recon = conjugate_gradient(Q, b, x, maxiter=30)

x_recon_image = x_recon.view(torch.complex64).reshape((n_x, n_y))

plt.figure()
plt.subplot(1, 2, 1)
plt.imshow(torch.abs(AHd_image).numpy().T, cmap="gray")
plt.title(f"Zero-filled (R={R})")
plt.subplot(1, 2, 2)
plt.imshow(torch.abs(x_recon_image).numpy().T, cmap="gray")
plt.title(f"CG Reconstruction (R={R})")
plt.tight_layout()
plt.show()

## Task 6: The Role of Acceleration in EPI
 
You have now completed the full pipeline for a single value of `R`. In this final task, you will explore what happens as `R` changes — and discover that in EPI, the choice of acceleration factor is not just about scan time, but about image quality.
 
Go back to the **System and Sequence Parameters** cell at the top and re-run the entire notebook for each of the following values:
 
- **R = 1** — no acceleration, full k-space acquired in a single shot
- **R = 2** — 2× acceleration
- **R = 4** — 4× acceleration
For each, observe the CG reconstruction and answer the following questions:
 
1. **R = 1:** What artifact do you see and where does it come from? Think about the total duration of the EPI readout train and what happens to the signal phase over that time.
2. **R = 2:** How does the image compare to R = 1? Why does shortening the readout train help?
3. **R = 4:** A new type of artifact appears. This is not caused by off-resonance but by the increased distance between acquired k-space lines. What does it look like, and how does it differ from the R = 1 artifact?
4. Based on your observations, what is the optimal acceleration factor for this sequence and phantom, and why is it a compromise?